# Superstore — Python Analysis (BQ6–BQ10 + Advanced)
**Dataset:** Sample_Superstore.csv


| File | Phạm vi |
|------|---------|
| `Superstore_SQL.sql` | BQ1–BQ5 |
| `Superstore_Python.ipynb` | BQ6–BQ10 + Advanced |


## 0. Setup & Load Data

In [12]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('/content/Sample - Superstore.csv', encoding='latin-1')
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.month

num_rows = len(df)
num_cols = df.shape[1]
min_order_date = df['Order Date'].min().strftime('%d/%m/%Y')
max_order_date = df['Order Date'].max().strftime('%d/%m/%Y')
unique_orders = df['Order ID'].nunique()
unique_customers = df['Customer ID'].nunique()
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
missing_values = df.isnull().sum().sum()

print(f"Số dòng: {num_rows:,}")
print(f"Số cột: {num_cols}")
print(f"Giai đoạn: {min_order_date} – {max_order_date}")
print(f"Đơn hàng unique: {unique_orders:,}")
print(f"Khách hàng unique: {unique_customers:,}")
print(f"Tổng doanh thu: ${total_sales:,.0f}")
print(f"Tổng lợi nhuận: ${total_profit:,.0f}")
print(f"Missing values: {missing_values}")

Số dòng: 9,994
Số cột: 23
Giai đoạn: 03/01/2014 – 30/12/2017
Đơn hàng unique: 5,009
Khách hàng unique: 793
Tổng doanh thu: $2,297,201
Tổng lợi nhuận: $286,397
Missing values: 0


---
## BQ6 — % đóng góp doanh thu Sub-Category trong Category

**Câu hỏi:** Mỗi Sub-Category chiếm bao nhiêu % doanh thu của Category chứa nó?

**Kết quả:**
- Chairs chiếm **44.3%** Furniture sales — lớn nhất nhóm
- Tables chiếm **27.9%** Furniture sales nhưng margin **-8.6%** → lỗ ròng -$17,725
- Phones chiếm **39.5%** Technology sales
- Copiers chỉ **17.9%** Technology sales nhưng margin **37.2%** → sinh lời nhất


In [13]:
# BQ6
bq6 = (df.groupby(['Category', 'Sub-Category'])
         .agg(sales=('Sales','sum'), profit=('Profit','sum'))
         .reset_index())

# transform('sum') = SUM() OVER (PARTITION BY Category)
bq6['cat_total']  = bq6.groupby('Category')['sales'].transform('sum')
bq6['pct_of_cat'] = (bq6['sales'] / bq6['cat_total'] * 100).round(1)
bq6['margin_pct'] = (bq6['profit'] / bq6['sales'] * 100).round(1)
bq6 = bq6.sort_values(['Category', 'pct_of_cat'], ascending=[True, False])

print(bq6[['Category','Sub-Category','sales','profit','pct_of_cat','margin_pct']]
      .rename(columns={'sales':'Sales','profit':'Profit',
                       'pct_of_cat':'% of Category','margin_pct':'Margin%'})
      .to_string(index=False))


       Category Sub-Category       Sales      Profit  % of Category  Margin%
      Furniture       Chairs 328449.1030  26590.1663           44.3      8.1
      Furniture       Tables 206965.5320 -17725.4811           27.9     -8.6
      Furniture    Bookcases 114879.9963  -3472.5560           15.5     -3.0
      Furniture  Furnishings  91705.1640  13059.1436           12.4     14.2
Office Supplies      Storage 223843.6080  21278.8264           31.1      9.5
Office Supplies      Binders 203412.7330  30221.7633           28.3     14.9
Office Supplies   Appliances 107532.1610  18138.0054           15.0     16.9
Office Supplies        Paper  78479.2060  34053.5693           10.9     43.4
Office Supplies     Supplies  46673.5380  -1189.0995            6.5     -2.5
Office Supplies          Art  27118.7920   6527.7870            3.8     24.1
Office Supplies    Envelopes  16476.4020   6964.1767            2.3     42.3
Office Supplies       Labels  12486.3120   5546.2540            1.7     44.4

---
## BQ7 — Doanh thu từng tháng 2017 vs 2016

**Câu hỏi:** Mỗi tháng năm 2017 tăng hay giảm so với cùng kỳ 2016?

**Tại sao Python thay SQL:**
SQL cần CTE + self-join (~15 dòng) hoặc `LAG() OVER (PARTITION BY month ORDER BY year)`.
Python: `groupby + unstack(level=0)` pivot year thành cột — 2 dòng.

**Kết quả thực:**
- Tháng 1: **+137.1%** — tăng mạnh nhất
- Tháng 8: **+102.9%** — tháng thứ 2 tăng hơn 100%
- Tháng 5: **-22.3%** — giảm mạnh nhất
- Tháng 12: **-13.6%** — cuối năm sụt (dù Q4 thường cao)
- Tổng 2017: $733,215 vs 2016: $609,206 → **+$124,009 (+20.4%)**


In [14]:
# BQ7 — pivot year thành cột thay cho self-join SQL
monthly = (df[df['Year'].isin([2016, 2017])]
           .groupby(['Year', 'Month'])['Sales']
           .sum()
           .unstack(level=0)   # Year → column: đây là lý do Python gọn hơn SQL
           .round(2))
monthly.columns = ['sales_2016', 'sales_2017']
monthly['diff']       = (monthly['sales_2017'] - monthly['sales_2016']).round(2)
monthly['growth_pct'] = (monthly['diff'] / monthly['sales_2016'] * 100).round(1)

MONTHS = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
          7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly.index = monthly.index.map(MONTHS)

print(monthly.to_string())
print(f"\nTổng 2017: ${monthly.sales_2017.sum():,.0f}")
print(f"Tổng 2016: ${monthly.sales_2016.sum():,.0f}")
print(f"Net diff : ${monthly.diff.sum():,.0f}  ({monthly.diff.sum()/monthly.sales_2016.sum()*100:.1f}%)")


       sales_2016  sales_2017      diff  growth_pct
Month                                              
Jan      18542.49    43971.37  25428.88       137.1
Feb      22978.82    20301.13  -2677.69       -11.7
Mar      51715.88    58872.35   7156.47        13.8
Apr      38750.04    36521.54  -2228.50        -5.8
May      56987.73    44261.11 -12726.62       -22.3
Jun      40344.53    52981.73  12637.20        31.3
Jul      39261.96    45264.42   6002.46        15.3
Aug      31115.37    63120.89  32005.52       102.9
Sep      73410.02    87866.65  14456.63        19.7
Oct      59687.74    77776.92  18089.18        30.3
Nov      79411.97   118447.82  39035.85        49.2
Dec      96999.04    83829.32 -13169.72       -13.6

Tổng 2017: $733,215
Tổng 2016: $609,206


AttributeError: 'function' object has no attribute 'sum'

---
## BQ8 — Khách hàng Loyal (mua ≥ 3 năm khác nhau)

**Câu hỏi:** Ai là khách hàng trung thành? VIP top sales nào đang thực sự sinh lời?

**Lưu ý:** SQL (`HAVING COUNT(DISTINCT YEAR) >= 3`) ngắn hơn ở phần filter.
Python cần thêm 1 bước `.loc[mask]` sau groupby.
→ File SQL Junior đã có BQ8 version T-SQL. File này tính thêm `vip_flag` để export cho Power BI.

**Kết quả thực:**
- **628 / 793** khách hàng loyal = **79.2%** — tỉ lệ cao
- Sean Miller: $25,043 sales nhưng **lỗ -$1,981** → "expensive VIP"
- Tamara Chand: $19,052 sales, **lãi $8,981** (margin 47.1%) → VIP thực sự
- 117 / 628 loyal customers đang có profit âm


In [ ]:
# BQ8
cust = (df.groupby(['Customer Name', 'Segment'])
          .agg(years_active=('Year',    'nunique'),
               total_orders=('Order ID','nunique'),
               total_sales =('Sales',   'sum'),
               total_profit=('Profit',  'sum'))
          .reset_index())
cust['margin_pct']   = (cust['total_profit'] / cust['total_sales'] * 100).round(1)
cust['avg_discount'] = (df.groupby('Customer Name')['Discount'].mean() * 100).round(1).values

loyal = cust[cust['years_active'] >= 3].sort_values('total_sales', ascending=False).copy()
loyal['vip_flag'] = loyal['total_profit'].apply(
    lambda p: 'Net Loss VIP'  if p < 0
         else 'Low Margin VIP' if p/loyal.loc[loyal.total_profit==p,'total_sales'].values[0] < 0.05
         else 'Profitable')
# Đơn giản hơn với apply trực tiếp:
loyal['vip_flag'] = loyal.apply(
    lambda r: 'Net Loss VIP'   if r.total_profit < 0
         else 'Low Margin VIP' if r.margin_pct   < 5
         else 'Profitable', axis=1)

print(f"Total customers : {len(cust)}")
print(f"Loyal (≥3 yrs)  : {len(loyal)} ({len(loyal)/len(cust):.1%})")
print(f"Net Loss VIPs   : {(loyal.vip_flag=='Net Loss VIP').sum()}")
print()
print("Top 15 Loyal by Sales:")
cols = ['Customer Name','Segment','years_active','total_orders',
        'total_sales','total_profit','margin_pct','vip_flag']
print(loyal.head(15)[cols].to_string(index=False))


---
## BQ9 — Sub-Category sinh lời cao nhất mỗi Segment

**Câu hỏi:** Mỗi Customer Segment nên tập trung đẩy Sub-Category nào?

**Kỹ năng:** `groupby + rank()` — tương đương `RANK() OVER (PARTITION BY Segment)` trong SQL

**Kết quả thực:**
- **Copiers** là #1 profit ở **cả 3 Segments** (margin 34–41%)
- **Home Office + Office Supplies** = margin **20.8%** — cao nhất toàn matrix
- Tables và Supplies lỗ ở tất cả Segments


In [ ]:
# BQ9
bq9 = (df.groupby(['Segment', 'Sub-Category', 'Category'])
         .agg(total_sales=('Sales','sum'), total_profit=('Profit','sum'))
         .reset_index())
bq9['margin_pct']  = (bq9['total_profit'] / bq9['total_sales'] * 100).round(1)
bq9['rank_in_seg'] = (bq9.groupby('Segment')['total_profit']
                         .rank(ascending=False, method='min')
                         .astype(int))
bq9 = bq9.sort_values(['Segment', 'rank_in_seg'])

print("=== Top 3 Sub-Categories per Segment ===")
print(bq9[bq9['rank_in_seg'] <= 3]
      [['Segment','rank_in_seg','Sub-Category','total_profit','margin_pct']]
      .to_string(index=False))

print("\n=== Loss Sub-Categories per Segment ===")
print(bq9[bq9['total_profit'] < 0]
      [['Segment','Sub-Category','total_profit','margin_pct']]
      .sort_values(['Segment','total_profit'])
      .to_string(index=False))


---
## BQ10 — Phân loại đơn hàng High / Medium / Low theo Region

**Câu hỏi:** Mỗi Region có phân bổ đơn hàng theo giá trị như thế nào?

**Tại sao Python:** `pd.cut(bins, labels)` = 1 dòng thay cho `CASE WHEN…WHEN…ELSE` trong SQL.
Thêm nhiều ngưỡng → chỉ thêm 1 phần tử trong `bins`, SQL phải thêm nguyên 1 `WHEN`.

**Kết quả thực:**
- Tất cả 4 Regions: Low (<$500) chiếm ~74% số đơn — majority
- West có nhiều High-value orders nhất (215 đơn)
- Central: tỉ lệ High orders 11% nhưng margin vẫn thấp nhất (7.9%)
  → vấn đề không nằm ở order size mà ở discount policy


In [ ]:
# BQ10
order_val = (df.groupby(['Order ID', 'Region'])['Sales']
               .sum().reset_index(name='order_sales'))

order_val['value_tier'] = pd.cut(
    order_val['order_sales'],
    bins=[-1, 500, 1000, 1e9],
    labels=['Low (<$500)', 'Medium ($500–1K)', 'High (>$1K)'])

bq10 = (order_val.groupby(['Region', 'value_tier'], observed=True)
                 .agg(total_orders=('Order ID','count'),
                      total_sales =('order_sales','sum'))
                 .reset_index())
bq10['pct_orders'] = (bq10['total_orders']
                      / bq10.groupby('Region')['total_orders'].transform('sum') * 100).round(1)
bq10['total_sales'] = bq10['total_sales'].round(2)

print(bq10.to_string(index=False))

# Pivot để dễ đọc
print("\n=== Pivot — Order Count ===")
print(bq10.pivot(index='Region', columns='value_tier', values='total_orders')
          .to_string())
print("\n=== Pivot — % of Region Orders ===")
print(bq10.pivot(index='Region', columns='value_tier', values='pct_orders')
          .to_string())


---
## Advanced — RFM Segmentation

**Tại sao Python thay DAX:**
DAX cần `RANKX + SWITCH + CALCULATE` cho R, F, M = **9 separate measures**.
Python: `pd.qcut()` × 3 dòng → score xong.

| Dimension | Định nghĩa | Score 5 = tốt nhất |
|-----------|-----------|-------------------|
| **R**ecency | Ngày kể từ lần mua cuối (snapshot 2018-01-01) | Mua gần nhất |
| **F**requency | Số đơn hàng unique | Mua nhiều nhất |
| **M**onetary | Tổng doanh thu | Chi tiêu nhiều nhất |

**Kết quả thực (793 customers):**
Champions 124 · Loyal 255 · Potential 217 · At Risk 118 · Lost 79


In [ ]:
# RFM Segmentation
SNAPSHOT = pd.Timestamp('2018-01-01')

rfm = (df.groupby('Customer Name')
         .agg(recency   =('Order Date', lambda x: (SNAPSHOT - x.max()).days),
              frequency  =('Order ID',   'nunique'),
              monetary   =('Sales',      'sum'))
         .round(2))

# pd.qcut() chia đều 5 nhóm theo phân vị — thay NTILE(5) + RANKX trong DAX
rfm['R'] = pd.qcut(rfm['recency'],              5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'],             5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R'] + rfm['F'] + rfm['M']

def rfm_label(s):
    if s >= 13: return 'Champions'
    if s >= 10: return 'Loyal'
    if s >= 7:  return 'Potential'
    if s >= 5:  return 'At Risk'
    return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(rfm_label)
rfm = rfm.reset_index()

# Summary
seg_summary = (rfm.groupby('Segment')
                  .agg(customers =('Customer Name','count'),
                       avg_recency  =('recency','mean'),
                       avg_frequency=('frequency','mean'),
                       avg_monetary =('monetary','mean'))
                  .round(1))
SEG_ORDER = ['Champions','Loyal','Potential','At Risk','Lost']
print("=== RFM Segment Summary ===")
print(seg_summary.reindex(SEG_ORDER).to_string())

print("\n=== Full RFM Table (top 20 by score) ===")
print(rfm.nlargest(20,'RFM_Score')
         [['Customer Name','recency','frequency','monetary','R','F','M','RFM_Score','Segment']]
         .to_string(index=False))


---
## Advanced — Cohort Retention Matrix

**Tại sao Python thay Power Query:**
Power Query cần 15+ bước (merge, group, conditional column, pivot thủ công).
Python: `groupby + unstack + div` = 4 dòng. Tự mở rộng khi có thêm năm dữ liệu.

**Đọc kết quả — Cohort 2014 (595 khách mua lần đầu năm 2014):**

| Năm | Active | Retention |
|-----|--------|-----------|
| 2014 | 595 | 100.0% |
| 2015 | 437 | 73.4% |
| 2016 | 485 | 81.5% |
| 2017 | 517 | 86.9% |

Retention tăng dần → nền tảng khách hàng ngày càng vững.


In [ ]:
# Cohort Retention
df['cohort_year'] = df.groupby('Customer Name')['Year'].transform('min')

cohort_counts = (df.groupby(['cohort_year', 'Year'])['Customer Name']
                   .nunique()
                   .unstack()
                   .fillna(0)
                   .astype(int))

cohort_size = cohort_counts.iloc[:, 0]
retention   = (cohort_counts.div(cohort_size, axis=0) * 100).round(1)

print("=== Cohort Active Customers (Count) ===")
print(cohort_counts.to_string())

print("\n=== Cohort Retention Rate (%) ===")
# Chỉ hiển thị Cohort 2014 vì các cohort sau chưa đủ follow-up years
print(retention.loc[[2014]].to_string())

print("\n=== Note ===")
print("Cohort 2015 (136 khách), 2016 (51 khách), 2017 (11 khách)")
print("— quá nhỏ để rút kết luận thống kê.")


---
## Advanced — Discount Anomaly Detection (Z-score)

**Tại sao Python thay DAX:**
DAX không có hàm z-score native. Cần `STDEVX.P(ALL(...)) + AVERAGEX` tính thủ công.
Python: `scipy.stats.zscore` = 1 dòng cho toàn cột.

**Logic:** Trong nhóm giao dịch đang lỗ, giao dịch nào có `|z-score| > 2`
là outlier discount thực sự — cần review ưu tiên.


In [ ]:
# Discount Anomaly Detection
df_loss = df[df['Profit'] < 0].copy()

# 1 dòng thay cho STDEVX.P + AVERAGEX trong DAX
df_loss['disc_zscore'] = np.abs(stats.zscore(df_loss['Discount']))
df_loss['margin_pct']  = (df_loss['Profit'] / df_loss['Sales'] * 100).round(1)

anomalies = df_loss[df_loss['disc_zscore'] > 2].sort_values('Profit')
normal    = df_loss[df_loss['disc_zscore'] <= 2]

print(f"Tổng giao dịch lỗ       : {len(df_loss):,}")
print(f"Anomaly (|z| > 2)       : {len(anomalies):,}")
print(f"Profit destroyed        : ${anomalies['Profit'].sum():,.0f}")
print(f"Disc trung bình — normal  : {normal['Discount'].mean():.1%}")
print(f"Disc trung bình — anomaly : {anomalies['Discount'].mean():.1%}")

print("\n=== Top 20 Anomalous Transactions (worst profit first) ===")
print(anomalies[['Order ID','Order Date','Customer Name','Category',
                  'Sub-Category','Sales','Discount','Profit','margin_pct']]
      .head(20).to_string(index=False))

print("\n=== Anomaly count by Sub-Category ===")
print(anomalies.groupby(['Category','Sub-Category'])
               .agg(count=('Profit','count'), total_loss=('Profit','sum'))
               .sort_values('total_loss')
               .to_string())


---
## Advanced — Customer Lifetime Value (CLV)

**Tại sao Python thay DAX:**
DAX tính CLV cần `AVERAGEX + DIVIDE + FILTER` lồng nhau per customer.
Python: pre-aggregate 3 metrics, nhân trực tiếp = 1 dòng vectorized.

**Formula:** `CLV = AOV × Frequency × Margin`
*(4-year simplified CLV — không dùng discount rate)*

| Metric | Định nghĩa |
|--------|-----------|
| AOV | Average Order Value (tính ở cấp order, không phải cấp row) |
| Frequency | Số đơn hàng trong 4 năm |
| Margin | Profit / Sales của customer đó |

**Kết quả thực:**
- Top CLV: Tamara Chand ($8,981), Raymond Buch ($6,976), Sanjit Chand ($5,757)
- **155 / 793 customers (19.5%)** có CLV âm → đang phá hủy giá trị


In [ ]:
# Customer Lifetime Value
# Bước 1: tính giá trị từng order (cấp order, không phải cấp row)
order_lvl = (df.groupby(['Customer Name', 'Order ID'])['Sales']
               .sum().reset_index(name='order_sales'))

clv_df = (order_lvl.groupby('Customer Name')
                   .agg(aov        =('order_sales','mean'),
                        frequency  =('Order ID',   'count'),
                        total_sales=('order_sales','sum'))
                   .reset_index())

# Bước 2: margin thực tế từng customer
margin = (df.groupby('Customer Name')
            .agg(profit=('Profit','sum'), sales=('Sales','sum'))
            .assign(margin=lambda x: x.profit / x.sales)
            .reset_index()[['Customer Name','margin','profit']])

seg_map = df.groupby('Customer Name')['Segment'].first().reset_index()
clv_df  = clv_df.merge(margin, on='Customer Name').merge(seg_map, on='Customer Name')

# Bước 3: CLV = AOV × Frequency × Margin — 1 dòng
clv_df['CLV'] = (clv_df['aov'] * clv_df['frequency'] * clv_df['margin']).round(2)
clv_df['clv_tier'] = clv_df['CLV'].apply(
    lambda v: 'High Value'      if v > 3000
         else 'Mid Value'       if v > 500
         else 'Low Value'       if v >= 0
         else 'Value Destroyer')
clv_df = clv_df.sort_values('CLV', ascending=False).reset_index(drop=True)

print(f"Negative CLV  : {(clv_df.CLV < 0).sum()} ({(clv_df.CLV < 0).mean():.1%})")
print(f"Total CLV pool: ${clv_df.CLV.sum():,.0f}")
print(f"Median CLV    : ${clv_df.CLV.median():,.0f}")

print("\n=== CLV Tier Distribution ===")
print(clv_df['clv_tier'].value_counts().to_string())

print("\n=== Top 20 Customers by CLV ===")
print(clv_df.head(20)
      [['Customer Name','Segment','aov','frequency','margin','CLV','clv_tier']]
      .assign(aov=lambda x: x.aov.round(0),
              margin=lambda x: (x.margin*100).round(1))
      .rename(columns={'aov':'AOV($)','margin':'Margin%'})
      .to_string(index=False))


---
## Export — Lưu kết quả để dùng trong Power BI

In [ ]:
# Chạy cell này để export tất cả kết quả ra CSV
# → Import vào Power BI bằng Get Data > Text/CSV

# BQ6
bq6.to_csv('export_bq6_subcat_pct.csv', index=False)

# BQ7
monthly.to_csv('export_bq7_yoy_monthly.csv')

# BQ8
loyal.to_csv('export_bq8_loyal_customers.csv', index=False)

# BQ9
bq9.to_csv('export_bq9_segment_subcat.csv', index=False)

# BQ10
bq10.to_csv('export_bq10_order_tier.csv', index=False)

# RFM
rfm.to_csv('export_rfm_segments.csv', index=False)

# CLV
clv_df.to_csv('export_clv.csv', index=False)

print("✓ Exported 7 files — import vào Power BI để build visual.")
print("  Xem Power_BI_Guide_BQ6_BQ10.md để biết cách dựng từng chart.")
